# **Cloud-Native GRIB2: Accessing S3 Data with grib2io and VirtualiZarr**

This notebook demonstrates how to efficiently access GRIB2 data stored on AWS S3 without streaming the entire file. We'll leverage pre-computed `.grib2ioidx` index files to build a virtual Xarray dataset.

## **The GRIB2 Challenge**

GRIB2 is a streaming format without a central metadata header. To create a virtual Zarr manifest, a standard scanner must read the file from start to finish to find message boundaries. For a multi-gigabyte file on S3, this normally means streaming all that data just to find the offsets.

### **The Solution: Sidecar Indices**
By using a sidecar index file (generated via `grib2io`), we can fetch the byte offsets and lengths directly, avoiding the need to scan the binary GRIB file.

In [ ]:
import xarray as xr
import grib2io
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
import s3fs
import json
import os

## **1. Locating GFS Data on S3**

We'll use a public GFS dataset from the NOAA Open Data Program on AWS.

In [ ]:
# Example S3 path for a GFS file
s3_bucket = "noaa-gfs-bdp-pds"
s3_path = "gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000"
s3_url = f"s3://{s3_bucket}/{s3_path}"
s3_idx_url = f"{s3_url}.grib2ioidx" # Pre-computed grib2io index

## **2. Building the Virtual Manifest via Index**

Instead of scanning the GRIB file, `grib2io` can use the remote `.grib2ioidx` file. In a production workflow, you would use `ReferenceGenerator` with a pre-loaded index.

In [ ]:
from grib2io.kerchunk import ReferenceGenerator

# In a real scenario, grib2io.open(s3_url, use_index=True) would 
# automatically find the .grib2ioidx file on S3.
gen = ReferenceGenerator(s3_url)

# The generator uses the index to map out the chunks instantly
manifest = gen.generate()

manifest_path = "gfs_s3_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)

## **3. Ingesting into VirtualiZarr**

Now we load the S3-backed manifest into VirtualiZarr.

In [ ]:
vds = open_virtual_dataset(
    manifest_path, 
    parser=KerchunkJSONParser(),
    loadable_variables=[]
)

display(vds)

## **4. High-Performance Access**

When you access data from this virtual dataset, Xarray makes targeted byte-range requests to S3, only pulling the specific GRIB messages required for your operation.

In [ ]:
# Example: Slice a small region of Temperature at the surface
# This only downloads the few hundred kilobytes for that specific message's grid.
if 'TMP' in vds.data_vars:
    temp_slice = vds.TMP.isel(y=slice(100, 200), x=slice(100, 200))
    display(temp_slice)